In [1]:
import os
import re
import sys
import json
import pickle
import joblib
import pandas as pd
import numpy as np
from collections import defaultdict
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.mixture import GaussianMixture
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score

# Ensure the root of project is in path
# In a notebook, we might need to adjust this depending on where it's run
base_dir = Path(os.getcwd()).resolve().parent
if str(base_dir / 'Datacleaning') not in sys.path:
    sys.path.insert(0, str(base_dir / 'Datacleaning'))

from header_feature_extractor import HeaderFeatureExtractor

In [2]:
def preprocess_body_text(text):
    """
    Preprocess body text based on USENIX Security '19 (sec19) baseline:
    - Remove salutations ('Dear', 'Hi', 'Hello')
    - Remove footers and common signatures ('Best regards', 'Sincerely', 'Thanks')
    - Convert to lowercase
    """
    if not isinstance(text, str):
        return ""
    
    # Normalize line endings
    text = text.replace('\r', '').lower()
    
    # Remove common salutations
    text = re.sub(r'^(?:hi|dear|hello|greetings|hey)(?:\s+\w+){0,3}[\s,]*\n+', '', text)
    
    # Remove common sign-offs and footers/signatures
    signoffs = r'(?:best regards|regards|sincerely|kind regards|thanks|thank you|cheers|best|respectfully)'
    text = re.sub(rf'{signoffs}[\s,:\-]*\n+.*$', '', text, flags=re.DOTALL)
    
    return text.strip()

In [3]:
print("==========================================================")
print("1. Loading Email_phishing.csv dataset...")
dataset_path = base_dir / 'Dataset' / 'Email_phishing.csv'
if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset not found at {dataset_path}")

df = pd.read_csv(dataset_path)
print(f"[OK] Loaded {len(df)} raw rows")

# Clean duplicates and NA

df = df.drop_duplicates().dropna(subset=['label', 'header', 'body']).copy()
df['label'] = df['label'].replace({'spam': 1, 'ham': 0})
df['label'] = pd.to_numeric(df['label'], errors='coerce')
df = df.dropna(subset=['label']).copy()
df['label'] = df['label'].astype(int)
print(
    f"[OK] Cleaned dataset: {len(df)} unique rows "
    f"(Phishing/Spam: {(df['label'] == 1).sum()}, Legitimate/Ham: {(df['label'] == 0).sum()})"
)

1. Loading Email_phishing.csv dataset...
[OK] Loaded 61426 raw rows
[OK] Cleaned dataset: 50336 unique rows (Phishing/Spam: 31859, Legitimate/Ham: 18477)


In [4]:
print("\n2. Extracting structured header features...")
extractor = HeaderFeatureExtractor()

# Pass 1: Learn sender and reply-to frequencies
for _, row in df.iterrows():
    sender = row.get('from', '')
    reply_to_match = re.search(
        r'Reply-To:\s*(.+?)(?:\n|$)', str(row.get('header', '')), re.IGNORECASE
    )
    reply_to = reply_to_match.group(1).strip() if reply_to_match else sender
    subject = row.get('subject', '')

    extractor.extract_header_features(sender, reply_to, subject, is_training=True)

print(f"[OK] Learned stats for {len(extractor.sender_frequency)} unique senders")

# Pass 2: Compute structured features
features_list = []
for _, row in df.iterrows():
    sender = row.get('from', '')
    reply_to_match = re.search(
        r'Reply-To:\s*(.+?)(?:\n|$)', str(row.get('header', '')), re.IGNORECASE
    )
    reply_to = reply_to_match.group(1).strip() if reply_to_match else sender
    subject = row.get('subject', '')

    feat = extractor.extract_header_features(sender, reply_to, subject, is_training=False)
    features_list.append(
        {
            'reply_to_mismatch': feat['reply_to_mismatch'],
            'name_email_mismatch': feat['name_email_mismatch'],
            'sender_rarity': feat['sender_rarity'],
            'impersonation_score': feat['impersonation_score'],
        }
    )

features_df = pd.DataFrame(features_list, index=df.index)
df = pd.concat([df, features_df], axis=1)
print("[OK] Extracted structured features: reply_to_mismatch, name_email_mismatch, sender_rarity")


2. Extracting structured header features...
[OK] Learned stats for 28502 unique senders
[OK] Extracted structured features: reply_to_mismatch, name_email_mismatch, sender_rarity


In [5]:
print("\n3. Performing GMM Clustering Under-sampling on Legitimate (ham) emails...")
df_ham = df[df['label'] == 0].copy()
df_spam = df[df['label'] == 1].copy()

# Fit Gaussian Mixture Model with 85 clusters on Legitimate features
X_ham_features = df_ham[['reply_to_mismatch', 'name_email_mismatch', 'sender_rarity']].values
n_clusters = 85
gmm = GaussianMixture(n_components=n_clusters, covariance_type='diag', random_state=42)
gmm.fit(X_ham_features)

# Predict clusters for each ham email
df_ham['cluster'] = gmm.predict(X_ham_features)

# Target under-sampled legitimate count

target_ham_count = 5000
cluster_counts = df_ham['cluster'].value_counts()

sampled_ham_dfs = []
for cluster_id, count in cluster_counts.items():
    sample_size = int(np.round((count / len(df_ham)) * target_ham_count))
    sample_size = max(1, min(sample_size, count))

    cluster_subset = df_ham[df_ham['cluster'] == cluster_id]
    sampled_subset = cluster_subset.sample(n=sample_size, random_state=42)
    sampled_ham_dfs.append(sampled_subset)

df_ham_sampled = pd.concat(sampled_ham_dfs, ignore_index=True)
print(
    f"[OK] Under-sampled legitimate class from {len(df_ham)} down to {len(df_ham_sampled)} rows "
    f"using {n_clusters} GMM clusters"
)

# Combine sampled legitimate with all phishing/spam emails
df_balanced = pd.concat([df_ham_sampled, df_spam], ignore_index=True)
print(
    f"[OK] Balanced dataset: {len(df_balanced)} rows "
    f"(Phishing: {len(df_spam)}, Legitimate: {len(df_ham_sampled)})"
)


3. Performing GMM Clustering Under-sampling on Legitimate (ham) emails...


C:\Users\Jay\AppData\Roaming\Python\Python314\site-packages\sklearn\base.py:1336: ConvergenceWarning: Number of distinct clusters (24) found smaller than n_clusters (85). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


[OK] Under-sampled legitimate class from 18477 down to 4999 rows using 85 GMM clusters
[OK] Balanced dataset: 36858 rows (Phishing: 31859, Legitimate: 4999)


In [6]:
print("\n4. Training Stage 1 Impersonation Classifier (Random Forest on Structured Features)...")
X_stage1 = df_balanced[['reply_to_mismatch', 'name_email_mismatch', 'sender_rarity']].values
y_stage1 = df_balanced['label'].values

X_train_s1, X_test_s1, y_train_s1, y_test_s1 = train_test_split(
    X_stage1, y_stage1, test_size=0.2, random_state=42, stratify=y_stage1
)

stage1_rf = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, n_jobs=-1)
stage1_rf.fit(X_train_s1, y_train_s1)

y_pred_s1 = stage1_rf.predict(X_test_s1)
print(f"[OK] Stage 1 Accuracy: {accuracy_score(y_test_s1, y_pred_s1):.4f}")
print("Stage 1 Impersonation Report:")
print(classification_report(y_test_s1, y_pred_s1))


4. Training Stage 1 Impersonation Classifier (Random Forest on Structured Features)...
[OK] Stage 1 Accuracy: 0.9405
Stage 1 Impersonation Report:
              precision    recall  f1-score   support

           0       0.73      0.90      0.80      1000
           1       0.98      0.95      0.96      6372

    accuracy                           0.94      7372
   macro avg       0.86      0.92      0.88      7372
weighted avg       0.95      0.94      0.94      7372



In [7]:
print("\n5. Applying Sequential Impersonation Gate (Stage 1)...")

df['stage1_prob'] = stage1_rf.predict_proba(
    df[['reply_to_mismatch', 'name_email_mismatch', 'sender_rarity']].values
)[:, 1]

impersonation_threshold = 0.5
df_gated = df[df['stage1_prob'] >= impersonation_threshold].copy()
print(
    f"[OK] Impersonation Gate: Passed {len(df_gated)} emails out of {len(df)} total emails "
    f"(Gated Phishing: {(df_gated['label'] == 1).sum()}, Gated Legitimate: {(df_gated['label'] == 0).sum()})"
)


5. Applying Sequential Impersonation Gate (Stage 1)...
[OK] Impersonation Gate: Passed 32210 emails out of 50336 total emails (Gated Phishing: 30239, Gated Legitimate: 1971)


In [8]:
print("\n6. Training Stage 2 Body Classifier (KNN on preprocessed body text)...")

# Preprocess bodies

df_gated['body_cleaned'] = df_gated['body'].apply(preprocess_body_text)

X_train_s2_df, X_test_s2_df, y_train_s2, y_test_s2 = train_test_split(
    df_gated,
    df_gated['label'].values,
    test_size=0.2,
    random_state=42,
    stratify=df_gated['label'].values,
)

body_vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2), stop_words='english')
X_train_body = body_vectorizer.fit_transform(X_train_s2_df['body_cleaned'])
X_test_body = body_vectorizer.transform(X_test_s2_df['body_cleaned'])

body_knn = KNeighborsClassifier(n_neighbors=5, n_jobs=-1)
body_knn.fit(X_train_body, y_train_s2)

y_pred_s2 = body_knn.predict(X_test_body)
print(f"[OK] Stage 2 KNN Body Accuracy: {accuracy_score(y_test_s2, y_pred_s2):.4f}")
print("Stage 2 KNN Body Report:")
print(classification_report(y_test_s2, y_pred_s2))


6. Training Stage 2 Body Classifier (KNN on preprocessed body text)...
[OK] Stage 2 KNN Body Accuracy: 0.9480
Stage 2 KNN Body Report:
              precision    recall  f1-score   support

           0       0.98      0.15      0.26       394
           1       0.95      1.00      0.97      6048

    accuracy                           0.95      6442
   macro avg       0.97      0.58      0.62      6442
weighted avg       0.95      0.95      0.93      6442



In [9]:
print("\n7. Fitting Stacking Classifier (Logistic Regression)...")

s1_test_probs = X_test_s2_df['stage1_prob'].values
s2_test_probs = body_knn.predict_proba(X_test_body)[:, 1]
X_stack_test = np.column_stack((s1_test_probs, s2_test_probs))

s1_train_probs = X_train_s2_df['stage1_prob'].values
s2_train_probs = body_knn.predict_proba(X_train_body)[:, 1]
X_stack_train = np.column_stack((s1_train_probs, s2_train_probs))

stacking_model = LogisticRegression()
stacking_model.fit(X_stack_train, y_train_s2)

final_preds = stacking_model.predict(X_stack_test)
final_acc = accuracy_score(y_test_s2, final_preds)
print(f"[OK] Stacking Model Accuracy: {final_acc:.4f}")
print("Stacked Model Report:")
print(classification_report(y_test_s2, final_preds))


7. Fitting Stacking Classifier (Logistic Regression)...
[OK] Stacking Model Accuracy: 0.9593
Stacked Model Report:
              precision    recall  f1-score   support

           0       0.90      0.38      0.53       394
           1       0.96      1.00      0.98      6048

    accuracy                           0.96      6442
   macro avg       0.93      0.69      0.76      6442
weighted avg       0.96      0.96      0.95      6442



In [10]:
print("\n8. Exporting models and vectorizers to extension backend...")
export_dir = base_dir / 'ExtensionApp' / 'backend' / 'models'
export_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(stage1_rf, export_dir / 'header_model.joblib')
joblib.dump(body_knn, export_dir / 'body_model.joblib')
joblib.dump(stacking_model, export_dir / 'final_model.joblib')
joblib.dump(body_vectorizer, export_dir / 'body_vectorizer.joblib')

metadata = {
    'total_legitimate_emails': len(df_ham),
    'total_phishing_emails': len(df_spam),
    'impersonation_threshold': impersonation_threshold,
    'best_header_model': 'RandomForest(Structured)',
    'best_body_model': 'KNN(PreprocessedText)',
    'gmm_components': n_clusters,
    'final_cascaded_accuracy': float(final_acc),
}

with open(export_dir / 'metadata.json', 'w') as f:
    json.dump(metadata, f, indent=2)

print(f"[OK] Saved files to {export_dir}:")
print("  - header_model.joblib (Stage 1 RF)")
print("  - body_model.joblib (Stage 2 KNN)")
print("  - final_model.joblib (Logistic Regression Stacker)")
print("  - body_vectorizer.joblib (TF-IDF Vectorizer)")
print("  - metadata.json")
print("\n[OK] Alignment with sec19 USENIX Security Baseline is successfully complete!")
print("==========================================================")


8. Exporting models and vectorizers to extension backend...
[OK] Saved files to C:\Users\Jay\Projects\AI-extension-BEC-detection\ExtensionApp\backend\models:
  - header_model.joblib (Stage 1 RF)
  - body_model.joblib (Stage 2 KNN)
  - final_model.joblib (Logistic Regression Stacker)
  - body_vectorizer.joblib (TF-IDF Vectorizer)
  - metadata.json

[OK] Alignment with sec19 USENIX Security Baseline is successfully complete!
